In [1]:
import openai
import langchain
import pinecone
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader,PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import Pinecone, PineconeVectorStore

/Users/suryaprkakash/Desktop/gen_ai_projects/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
## Read the document
def read_doc(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [4]:
doc = read_doc("LLM Generic app/document")
len(doc)

0

In [5]:
## Divide the docs into chunks
def chunk_data(docs,chunk_size=700,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return doc

In [6]:
chunkDoc = chunk_data(doc)
chunkDoc[50]

IndexError: list index out of range

In [ ]:
#Embedding technique
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

In [ ]:
vectors = embeddings.embed_query("hello world")
vectors[1:10]

In [ ]:
from pinecone import Pinecone

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

index = pc.Index("langchainvectordb")

In [ ]:
index

In [ ]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [ ]:
vectorstore.add_documents(chunkDoc)

In [ ]:
#Cosine similarity retrieve results from vectorDB
def retrieve_query(query, k=2):
    matching_results = vectorstore.similarity_search(query,k=k)
    return matching_results

In [ ]:
from langchain_groq import ChatGroq
from langchain_openai import OpenAI
from langchain_classic.chains.question_answering import load_qa_chain
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5
)
chain = load_qa_chain(llm,chain_type='stuff')

In [ ]:
def retrieve_answers(query):
    doc_search = retrieve_query(query)
    print(doc_search)
    response=chain.run(input_documents = doc_search,question=query)
    return response

In [ ]:
our_query = "How much the agriculture target will be increased by how many crores?"
answer = retrieve_answers(our_query)

print(answer)